# Causal interventions on the belief subspace (Sections 6 and 7)

**Paper section.** Section 6 (Controlling representations via the belief state) with Figure 6 and Appendices M, N, and O with Figures 53 to 70. Section 7 (Controlling predictions via the belief state) with Figure 7 and the right panel of Figure 1 (b), Appendix E with Figure 16, and Appendix P with Figures 71 to 76.

**Claim.** In the paper's words, "Steering the belief state systematically changes log-NTP decodability", "Belief interventions have privileged effects beyond the NTP subspace", "Belief state interventions systematically shift predictions", and "The injected belief largely overrides conflicting context."

**Experiment.** Both experiments come from `scripts/run_interventions.sh`. Section 6 steers the belief subspace at one layer over the last 5,000 positions toward past-inconsistent targets, along the NTP-preserving delta direction, and along a random direction of matched magnitude, and decodes with probes fit before the steer at every later layer. Section 7 patches or steers the belief subspace at the last k positions (k = 1, 5, 10) and measures the KL from the injected belief's NTP to the model's prediction.

**Saved Outputs.** In `figures/`: the Section 6 maps, strips, and summaries named `xmap_*`, `ker_*`, `randdir_*`, and `delta_vs_random_summary_*` with the suffix `_<model>_rsplit.pdf`, and the Section 7 figures `intervene_main_patch.pdf`, `intervene_main_steer.pdf`, `intervene_all_<model>.pdf`, and `fig1_intervene_wing.svg`.

**How the notebook works.** The first code cell sets the paths. The Section 6 cells read `results/belief_steering_<donor>_<model>.csv` for the three donors, build the (steering layer, readout layer) maps of the frozen-probe R², and write Figure 6 and the appendix summaries. The Section 7 cells read `results/prediction_interventions_<model>.csv`, average the KL over seeds and layers, and write Figures 7 and 16, the Figure 1 panel, and the Appendix P summaries to `figures/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys
from matplotlib.lines import Line2D
from scipy.stats import pearsonr, spearmanr
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.ticker import MultipleLocator

# ═══════════════════════════════════════════════════════════
# CONFIG — change these paths, everything else follows
# ═══════════════════════════════════════════════════════════
RESULTS_DIR = '../results'
PLOT_DIR = '../figures'
os.makedirs(PLOT_DIR, exist_ok=True)

MODEL_KEYS = ['qwen35_9b', 'qwen35_4b', 'llama_31_8b', 'llama_32_3b', 'gemma_4_e4b', 'gemma_4_e2b']
MODEL_LABELS = {
    'qwen35_9b': 'Qwen 3.5 9B', 'qwen35_4b': 'Qwen 3.5 4B',
    'llama_31_8b': 'Llama 3.1 8B', 'llama_32_3b': 'Llama 3.2 3B',
    'gemma_4_e4b': 'Gemma 4 E4B', 'gemma_4_e2b': 'Gemma 4 E2B',
}

# Representative (HMM, param) for single-param plots
TARGETS = [
    ('Mess3', 'a=0.01, x=0.02'),
    # ('Arch', 'a=0.9'),
    ('Arch', 'a=0.99'),
    ('Wing', 'a=0.98, x=0.4'),
    ('Strata', 'a=0.97, t0=0.38, t1=0.54'),
]
HMM_ORDER = ['Mess3', 'Arch', 'Wing', 'Strata']
MODEL_ORDER = list(MODEL_KEYS)
HMM_COLORS = {'Mess3': 'tab:blue', 'Arch': 'tab:orange', 'Wing': 'tab:green', 'Strata': 'tab:red'}
HMMS = ['Mess3', 'Arch', 'Wing', 'Strata']

# ═══════════════════════════════════════════════════════════
# Style
# ═══════════════════════════════════════════════════════════
matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans'],
    'font.weight': 'light',
})
sns.set_context('notebook')
tab10 = sns.color_palette('tab10')

def fmt_tick(v):
    s = f'{v:.1f}' if v == int(v) else f'{v:.2f}'
    if -1 < v < 1 and '.' in s:
        s = s.replace('0.', '.')   # 0.97 → .97  AND  -0.45 → -.45
    return s

# ═══════════════════════════════════════════════════════════
# Loaders
# ═══════════════════════════════════════════════════════════
def load_csv(prefix, model_key):
    path = os.path.join(RESULTS_DIR, f'{prefix}_{model_key}.csv')
    if not os.path.exists(path):
        print(f'  MISSING: {path}')
        return None
    return pd.read_csv(path)

def load_npz(prefix, model_key):
    path = os.path.join(RESULTS_DIR, f'{prefix}_{model_key}.npz')
    if not os.path.exists(path):
        print(f'  MISSING: {path}')
        return None
    return np.load(path, allow_pickle=True)

def harmonize_cross(df):
    if df is None: return None
    if 'source' in df.columns:
        df = df.rename(columns={'source': 'train', 'target': 'test'})
    return df

def harmonize_wing_labels(cross_df, gt_df):
    if cross_df is None or gt_df is None: return cross_df, gt_df
    for name in ['Wing']:
        c_params = set(cross_df[cross_df['hmm']==name]['train'].unique()) if name in cross_df['hmm'].values else set()
        g_params = set(gt_df[gt_df['hmm']==name]['train'].unique()) if name in gt_df['hmm'].values else set()
        if c_params and g_params and c_params != g_params:
            mask = gt_df['hmm'] == name
            for col in ['train', 'test']:
                gt_df.loc[mask, col] = (gt_df.loc[mask, col]
                    .str.replace('x=', 'ALPHA=').str.replace('y=', 'x=').str.replace('ALPHA=', 'a='))
    return cross_df, gt_df

# ═══════════════════════════════════════════════════════════
# Auto-load all R² files
# ═══════════════════════════════════════════════════════════
r2_files = {}
for mk in MODEL_KEYS:
    df = load_csv('r2', mk)
    if df is not None:
        r2_files[mk] = df
        print(f'{MODEL_LABELS[mk]}: {len(df)} rows, HMMs={sorted(df["hmm"].unique())}')
print(f'\n{len(r2_files)} models loaded')

## Section 6: steering the belief state and decoding with frozen probes (Figure 6, Figures 53 to 70)

In [ ]:
# ── Section 6 data: belief steering with the paper's steering vector and the interleaved 20/80 probe split (scripts/run_interventions.sh) ──
import os, pandas as pd, numpy as np
SPLIT = 'random'; TAG = '_rsplit'     # figure-filename tag, kept so the file names match the paper's
_STORY = {'past_inconsistent': f'{RESULTS_DIR}/belief_steering_past_inconsistent_{{mk}}.csv',
          'ntp_matched':       f'{RESULTS_DIR}/belief_steering_ntp_matched_{{mk}}.csv',
          'random_matched':    f'{RESULTS_DIR}/belief_steering_random_matched_{{mk}}.csv'}
def story_path(donor, mk):
    return _STORY[donor].format(mk=mk)
S6_PLOT_DIR = PLOT_DIR
print(f'figure tag = {TAG!r}, figures -> {S6_PLOT_DIR}/')


In [ ]:
# ── Load the belief-steering CSVs (past-inconsistent donor) ──
_USECOLS = ['hmm','param','intervene_layer','readout_layer','phase','probe','ref','target_kind','value']

def load_destroy(model_key):
    path = story_path('past_inconsistent', model_key)
    if not os.path.exists(path):
        print(f'  MISSING: {path}'); return None
    df = pd.read_csv(path, usecols=_USECOLS)
    df['value'] = df['value'].clip(-1, 1).astype('float32')
    return df

destroy_dfs = {'belief': {}}
for mk in MODEL_KEYS:
    df = load_destroy(mk)
    if df is not None:
        destroy_dfs['belief'][mk] = df
        print(f'belief  {MODEL_LABELS[mk]}: {len(df):>10,} rows')
print(f"\nbelief: {len(destroy_dfs['belief'])}/{len(MODEL_KEYS)} files")


In [ ]:
# ── Section-6 plotting functions (paths via story_path; every save carries TAG and goes to S6_PLOT_DIR) ──
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.ticker import MultipleLocator, FuncFormatter
import matplotlib.transforms as mtransforms
import inspect

HMMS_KER = ['Arch', 'Wing', 'Strata']   # Mess3 has no ker donor (full-rank M)
PANEL_TITLE_FS = 34                       # process-name text above each panel
TICK_FS = 26                              # tick numbers on the main-text heatmaps
XLABEL_FS = 20                            # shared x-label size (strips)
YLABEL_FS = 20                            # y-label size (default)
LOGNTP_LABEL_FS = 32                      # x-label size on the 4-panel log-NTP maps
LOGNTP_YLABEL_FS = 24                     # y-label size on the 4-panel log-NTP maps (smaller so it fits)
BOTTOM_LABEL_FS = 32                      # x- and y-label size on the δ and v maps
YLABEL_PAD = 14                           # gap between y-label and tick numbers
STRIP_XLABEL_Y = -0.18                    # shared x-label position for the 2-inch-tall before-strips
HMAP_H = 6.4                              # main-text heatmap figure height: canvas room for the overflowing y-label
HMAP_WSPACE = 0.35                        # horizontal gap between heatmap panels
HMAP_LEFT = 0.15                          # left margin reserved for the y-label
HMAP_TOP, HMAP_BOTTOM = 0.80, 0.22        # vertical margins (canvas room above/below the panels)
XLABEL_GAP_IN = 1.15                      # shared x-label sits this many inches below the panel bottoms
CB_TICKS = [-1, -0.5, 0, 0.5, 1]          # colorbar ticks on the main-text maps
CB_TICK_FS = 26; CB_LABEL_FS = 30         # colorbar tick / label font sizes on the main-text maps
CB_GAP_IN = 0.45                          # horizontal gap between the last heatmap and the colorbar (inches)
CB_W_IN = 0.22                            # colorbar width (inches)
DELTA_XLABEL = r'$\delta$-probing layer';  DELTA_YLABEL = r'$\delta$-steering layer'
RAND_XLABEL  = r'$v$-probing layer';       RAND_YLABEL  = r'$v$-steering layer'

def _cmap():
    c = plt.get_cmap('RdBu'); c.set_bad('0.82'); return c

def _cb_formatter():
    """Colorbar ticks with one digit: 1, .5, 0, -.5, -1 (no leading/trailing zeros; plain '0')."""
    def _one_digit(x, pos=None):
        if abs(x) < 1e-9: return '0'
        s = f'{x:.1f}'.rstrip('0').rstrip('.')     # 1.0 -> '1', 0.5 -> '0.5', -0.5 -> '-0.5'
        return s.replace('0.', '.')                # '0.5' -> '.5', '-0.5' -> '-.5'
    return FuncFormatter(_one_digit)

def _triangle(ax, a, i, ylabel, vmax, hmm, cmap, ylabel_fs=None, tick_fs=None):
    """Shared (steering layer × readout layer) heatmap styling. a: Series indexed (L, r)."""
    Ls = sorted({L for L,_ in a.index}); Rs = sorted({r for _,r in a.index})
    li = {L:j for j,L in enumerate(Ls)}; ri = {r:j for j,r in enumerate(Rs)}
    G = np.full((len(Ls), len(Rs)), np.nan)
    for (L,r), v in a.items():
        G[li[L], ri[r]] = np.clip(v, -1, 1)
    off = np.array([G[li[L], ri[r]] for (L,r) in a.index if r>=L+2])
    print(f"    {hmm:7s} off-diag mean {np.nanmean(off):+.3f}   min {np.nanmin(off):+.2f}   %>0 {100*np.nanmean(off>0):.1f}%")
    ax.imshow(G, origin='lower', aspect='auto', cmap=cmap, norm=Normalize(-vmax, vmax),
              extent=[Rs[0]-.5, Rs[-1]+.5, Ls[0]-.5, Ls[-1]+.5])
    ax.set_box_aspect(1); ax.set_title(hmm, fontsize=PANEL_TITLE_FS, pad=10)
    ax.set_xticks([0,10,20,30]); ax.set_yticks([0,10,20,30])
    ax.xaxis.set_minor_locator(MultipleLocator(5)); ax.yaxis.set_minor_locator(MultipleLocator(5))
    ax.tick_params(which='major', labelsize=tick_fs or 18, length=4); ax.tick_params(which='minor', length=2)
    ax.set_axisbelow(False)
    ax.grid(True, which='major', color='0.6', alpha=0.4, lw=0.6)
    ax.grid(True, which='minor', color='0.6', alpha=0.25, lw=0.4)
    if i==0: ax.set_ylabel(ylabel, fontsize=ylabel_fs or YLABEL_FS, labelpad=YLABEL_PAD)

def _strip(ax, cl, vmax, hmm, cmap):
    """Single-row strip: clean-probe decodability vs probing layer (no steering dependence)."""
    Rs = sorted(cl.index); G = np.array([[np.clip(cl[r], -1, 1) for r in Rs]])
    print(f"    {hmm:7s} before: min {cl.min():+.2f}  max {cl.max():+.2f}  %>0 {100*(cl>0).mean():.0f}%")
    ax.imshow(G, origin='lower', aspect='auto', cmap=cmap, norm=Normalize(-vmax, vmax),
              extent=[Rs[0]-.5, Rs[-1]+.5, -0.5, 0.5])
    ax.set_title(hmm, fontsize=18, pad=8)
    ax.set_xticks([0,10,20,30]); ax.set_yticks([]); ax.xaxis.set_minor_locator(MultipleLocator(5))
    ax.tick_params(which='major', labelsize=14, length=4); ax.tick_params(which='minor', length=2)

def _colorbar(fig, axes, vmax, cmap, fraction, label='$R^2$', ticks=None, tick_fs=12, label_fs=15):
    sm = ScalarMappable(cmap=cmap, norm=Normalize(-vmax, vmax)); sm.set_array([])
    cb = fig.colorbar(sm, ax=list(axes), fraction=fraction, pad=0.01, ticks=ticks)
    cb.set_label(label, fontsize=label_fs); cb.ax.tick_params(labelsize=tick_fs)

def _main_layout(fig):
    fig.subplots_adjust(left=HMAP_LEFT, wspace=HMAP_WSPACE, top=HMAP_TOP, bottom=HMAP_BOTTOM)

def _main_colorbar(fig, axes, vmax, cmap, label='$R^2$'):
    """Colorbar as its own axis, pinned to the last panel's drawn box: same height, fixed gap."""
    fig.draw_without_rendering()
    pos = [ax for ax in axes if ax.get_visible()][-1].get_position()
    fw = fig.get_figwidth()
    cax = fig.add_axes([pos.x1 + CB_GAP_IN / fw, pos.y0, CB_W_IN / fw, pos.height])
    sm = ScalarMappable(cmap=cmap, norm=Normalize(-vmax, vmax)); sm.set_array([])
    cb = fig.colorbar(sm, cax=cax, ticks=CB_TICKS, format=_cb_formatter())
    cb.set_label(label, fontsize=CB_LABEL_FS); cb.ax.tick_params(labelsize=CB_TICK_FS)

def _main_xlabel(fig, axes, text, fs):
    """Shared x-label a fixed distance below the drawn panel bottoms (independent of the margins)."""
    fig.draw_without_rendering()
    y0 = min(ax.get_position().y0 for ax in axes if ax.get_visible())
    fig.supxlabel(text, fontsize=fs, y=y0 - XLABEL_GAP_IN / fig.get_figheight())

# ══ log-NTP (belief steer) ══
def logntp_before_strip(df, ref, mk, vmax=1.0):
    bef = df[(df.target_kind=='log_ntp')&(df.hmm!='Spiral')&(df.phase=='before')&(df.ref==ref)&(df.probe=='clean')]
    lab, tag = ('target', 'dtarget') if ref == 'new' else ('original', 'doriginal')
    cmap = _cmap(); fig, axes = plt.subplots(1, len(TARGETS), figsize=(16, 2.0))
    for i,(hmm,param) in enumerate(TARGETS):
        cl = bef[(bef.hmm==hmm)&(bef.param==param)].groupby('readout_layer')['value'].mean()
        if len(cl)==0: axes[i].set_visible(False); print(f'    {hmm:7s} MISSING (no rows in file)'); continue
        _strip(axes[i], cl, vmax, hmm, cmap)
    fig.supxlabel('log-NTP probing layer', fontsize=XLABEL_FS, y=STRIP_XLABEL_Y)
    fig.suptitle(f'{lab.capitalize()} log-NTP decodability before steering', fontsize=18, y=1.14)
    _colorbar(fig, axes, vmax, cmap, 0.022)
    fig.savefig(f'{S6_PLOT_DIR}/xmap_{tag}_logntp_before_{mk}{TAG}.pdf', bbox_inches='tight')
    plt.show(); plt.close()

def logntp_after_map(df, ref, mk, probe='frozen', vmax=1.0):
    aft = df[(df.target_kind=='log_ntp')&(df.hmm!='Spiral')&(df.phase=='after')&(df.ref==ref)&(df.probe==probe)]
    lab, tag = ('target', 'dtarget') if ref == 'new' else ('original', 'doriginal')
    cmap = _cmap(); fig, axes = plt.subplots(1, len(TARGETS), figsize=(16, HMAP_H))
    _main_layout(fig)
    for i,(hmm,param) in enumerate(TARGETS):
        a = aft[(aft.hmm==hmm)&(aft.param==param)].groupby(['intervene_layer','readout_layer'])['value'].mean()
        if len(a)==0: axes[i].set_visible(False); print(f'    {hmm:7s} MISSING (no rows in file)'); continue
        _triangle(axes[i], a, i, 'Belief steering layer', vmax, hmm, cmap, ylabel_fs=LOGNTP_YLABEL_FS, tick_fs=TICK_FS)
    _main_colorbar(fig, axes, vmax, cmap)
    _main_xlabel(fig, axes, 'log-NTP probing layer', LOGNTP_LABEL_FS)
    if probe != 'frozen':   # main-text (frozen) maps carry no title; appendix retrain maps keep theirs
        fig.suptitle(f'{lab.capitalize()} log-NTP decodability after belief steering ({probe})', fontsize=26, y=0.98)
    fig.savefig(f'{S6_PLOT_DIR}/xmap_{tag}_logntp_after_{probe}_{mk}{TAG}.pdf', bbox_inches='tight')
    plt.show(); plt.close()

# ══ δ coordinate (δ steer, frozen probe) ══
def delta_before_strip(mk, ref='new', vmax=1.0):
    f = story_path('ntp_matched', mk)
    if not os.path.exists(f): print('  MISSING:', f); return
    k = pd.read_csv(f); k['value'] = k['value'].clip(-1, 1)
    bef = k[(k.target_kind=='hidden') & (k.phase=='before') & (k.ref==ref) & (k.probe=='clean')]
    lab, tag = ('target', 'dtarget') if ref == 'new' else ('original', 'dorig')
    cmap = _cmap(); fig, axes = plt.subplots(1, len(HMMS_KER), figsize=(12, 2.0))
    for i, (ax, hmm) in enumerate(zip(axes, HMMS_KER)):
        cl = bef[bef.hmm==hmm].groupby('readout_layer')['value'].mean()
        _strip(ax, cl, vmax, hmm, cmap)
    fig.supxlabel(DELTA_XLABEL, fontsize=XLABEL_FS, y=STRIP_XLABEL_Y)
    fig.suptitle(f'{lab.capitalize()} δ-belief decodability before steering', fontsize=18, y=1.14)
    _colorbar(fig, axes, vmax, cmap, 0.03)
    fig.savefig(f'{S6_PLOT_DIR}/ker_{tag}_hidden_before_{mk}{TAG}.pdf', bbox_inches='tight')
    plt.show(); plt.close()

def delta_after_map(mk, ref='new', vmax=1.0):
    f = story_path('ntp_matched', mk)
    if not os.path.exists(f): print('  MISSING:', f); return
    k = pd.read_csv(f); k['value'] = k['value'].clip(-1, 1)
    aft = k[(k.target_kind=='hidden') & (k.phase=='after') & (k.ref==ref) & (k.probe=='frozen')]
    lab, tag = ('target', 'dtarget') if ref == 'new' else ('original', 'dorig')
    cmap = _cmap(); fig, axes = plt.subplots(1, len(HMMS_KER), figsize=(12, HMAP_H))
    _main_layout(fig)
    for i, (ax, hmm) in enumerate(zip(axes, HMMS_KER)):
        a = aft[aft.hmm==hmm].groupby(['intervene_layer','readout_layer'])['value'].mean()
        if len(a)==0: ax.set_visible(False); continue
        _triangle(ax, a, i, DELTA_YLABEL, vmax, hmm, cmap, ylabel_fs=BOTTOM_LABEL_FS, tick_fs=TICK_FS)
    _main_xlabel(fig, axes, DELTA_XLABEL, BOTTOM_LABEL_FS)
    # no title, no colorbar (the v-map to its right carries the shared legend)
    fig.savefig(f'{S6_PLOT_DIR}/ker_{tag}_hidden_after_{mk}{TAG}.pdf', bbox_inches='tight')
    plt.show(); plt.close()

# ══ random component (random steer, fixed projection onto v) ══
def random_after_map(mk, ref='new', vmax=1.0):
    f = story_path('random_matched', mk)
    if not os.path.exists(f): print('  MISSING:', f); return
    k = pd.read_csv(f); k['value'] = k['value'].clip(-1, 1)
    aft = k[(k.probe=='fixed_dir') & (k.target_kind=='rand_proj') & (k.ref==ref)]
    lab, tag = ('Target', 'dtarget') if ref == 'new' else ('Original', 'dorig')
    cmap = _cmap(); fig, axes = plt.subplots(1, len(HMMS_KER), figsize=(12, HMAP_H))
    _main_layout(fig)
    for i, (ax, hmm) in enumerate(zip(axes, HMMS_KER)):
        a = aft[aft.hmm==hmm].groupby(['intervene_layer','readout_layer'])['value'].mean()
        if len(a)==0: ax.set_visible(False); continue
        _triangle(ax, a, i, RAND_YLABEL, vmax, hmm, cmap, ylabel_fs=BOTTOM_LABEL_FS, tick_fs=TICK_FS)
    _main_colorbar(fig, axes, vmax, cmap)
    _main_xlabel(fig, axes, RAND_XLABEL, BOTTOM_LABEL_FS)
    # no title; shared legend for the bottom row, fewer ticks, larger text
    fig.savefig(f'{S6_PLOT_DIR}/randdir_{tag}_{mk}{TAG}.pdf', bbox_inches='tight')
    plt.show(); plt.close()

# ══ appendix per-model summary: all HMM parametrizations × probing layer, raw after-steering R² ══
def raw_profile_fig(df, mk, target_kind, fam_order, panel_word, xlabel, save_tag, vmax=1.0, probe='frozen', title_note=''):
    d = df[df.target_kind==target_kind]
    aft = (d[(d.phase=='after')&(d.probe==probe)&(d.intervene_layer>=0)]
           .groupby(['hmm','param','ref','intervene_layer','readout_layer'])['value']
           .mean().clip(-1,1).reset_index(name='va'))
    aft = aft[aft.readout_layer >= aft.intervene_layer + 2]
    t = aft.groupby(['hmm','param','ref','readout_layer'])['va'].mean()
    rows = [(fam, p) for fam in fam_order for p in sorted(d[d.hmm==fam]['param'].unique())]
    if not rows: return
    Rs = sorted(t.index.get_level_values('readout_layer').unique())
    fam_sizes = [sum(1 for f,_ in rows if f==fam) for fam in fam_order]; fam_edges = np.cumsum(fam_sizes)
    cmap = _cmap(); P = {}
    for ref in ['new','orig']:
        P[ref] = np.full((len(rows), len(Rs)), np.nan)
        for i,(fam,p) in enumerate(rows):
            for j,r in enumerate(Rs):
                P[ref][i,j] = t.get((fam, p, ref, r), np.nan)
        print(f"{MODEL_LABELS[mk]:14s} {target_kind:9s} {probe:9s} ref={ref:4s}  mean {np.nanmean(P[ref]):+.3f}   "
              f"min {np.nanmin(P[ref]):+.2f}   %>0 {100*np.nanmean(P[ref]>0):.1f}%")
    fig, axes = plt.subplots(1, 2, figsize=(13, 11 if len(rows)>35 else 9), sharey=True)
    for ax, ref, ttl in zip(axes, ['new','orig'], [f"Target {panel_word}", f"Original {panel_word}"]):
        ax.imshow(P[ref], aspect='auto', cmap=cmap, norm=Normalize(-vmax, vmax),
                  extent=[Rs[0]-.5, Rs[-1]+.5, len(rows)-.5, -.5])
        for e in fam_edges[:-1]:
            ax.axhline(e - 0.5, color='k', lw=1.2)
        ax.set_xticks([0, 10, 20, 30]); ax.set_xlabel(xlabel, fontsize=16)
        ax.set_title(ttl, fontsize=17, pad=10); ax.tick_params(labelsize=13, length=3)
    axes[0].set_yticks(range(len(rows))); axes[0].set_yticklabels([p for _,p in rows], fontsize=7)
    _tr = mtransforms.blended_transform_factory(axes[0].transAxes, axes[0].transData)
    for fam, e, s in zip(fam_order, fam_edges, fam_sizes):
        axes[0].text(-0.35, e - s/2 - 0.5, fam, fontsize=14, rotation=90, va='center', ha='center', transform=_tr)
    fig.suptitle(f'{MODEL_LABELS[mk]}{title_note}', fontsize=20, y=0.95)
    sm = ScalarMappable(cmap=cmap, norm=Normalize(-vmax, vmax)); sm.set_array([])
    cb = fig.colorbar(sm, ax=list(axes), fraction=0.025, pad=0.02)
    cb.set_label('$R^2$ after steering, mean over steering layers', fontsize=13); cb.ax.tick_params(labelsize=12)
    fig.savefig(f'{S6_PLOT_DIR}/{save_tag}_{mk}{TAG}.pdf', bbox_inches='tight')
    plt.show(); plt.close()

# ══ side-by-side all-HMM summary: δ after δ steering (left) vs random component after random steering (right) ══
def _profile_matrix(df, target_kind, probe, ref, rows):
    d = df[df.target_kind==target_kind]
    aft = (d[(d.phase=='after')&(d.probe==probe)&(d.ref==ref)&(d.intervene_layer>=0)]
           .groupby(['hmm','param','intervene_layer','readout_layer'])['value']
           .mean().clip(-1,1).reset_index(name='va'))
    aft = aft[aft.readout_layer >= aft.intervene_layer + 2]
    t = aft.groupby(['hmm','param','readout_layer'])['va'].mean()
    Rs = sorted(t.index.get_level_values('readout_layer').unique())
    P = np.full((len(rows), len(Rs)), np.nan)
    for i,(fam,p) in enumerate(rows):
        for j,r in enumerate(Rs):
            P[i,j] = t.get((fam, p, r), np.nan)
    return P, Rs

def delta_vs_random_summary(ker, rnd, mk, ref='new', vmax=1.0):
    rows = [(fam, p) for fam in HMMS_KER for p in sorted(ker[ker.hmm==fam]['param'].unique())]
    if not rows: return
    lab = 'Target' if ref == 'new' else 'Original'
    panels = [(ker, 'hidden',    'frozen',    f'{lab} δ-belief coordinate after δ steering',    DELTA_XLABEL),
              (rnd, 'rand_proj', 'fixed_dir', f'{lab} random component after random steering', RAND_XLABEL)]
    fam_sizes = [sum(1 for f,_ in rows if f==fam) for fam in HMMS_KER]; fam_edges = np.cumsum(fam_sizes)
    cmap = _cmap(); fig, axes = plt.subplots(1, 2, figsize=(13, 9), sharey=True)
    for ax, (df, tk, probe, ttl, xl) in zip(axes, panels):
        P, Rs = _profile_matrix(df, tk, probe, ref, rows)
        print(f"{MODEL_LABELS[mk]:14s} {tk:9s} {probe:9s} ref={ref:4s}  mean {np.nanmean(P):+.3f}   min {np.nanmin(P):+.2f}   %>0 {100*np.nanmean(P>0):.1f}%")
        ax.imshow(P, aspect='auto', cmap=cmap, norm=Normalize(-vmax, vmax),
                  extent=[Rs[0]-.5, Rs[-1]+.5, len(rows)-.5, -.5])
        for e in fam_edges[:-1]:
            ax.axhline(e - 0.5, color='k', lw=1.2)
        ax.set_xticks([0, 10, 20, 30]); ax.set_xlabel(xl, fontsize=16)
        ax.set_title(ttl, fontsize=15, pad=10); ax.tick_params(labelsize=13, length=3)
    axes[0].set_yticks(range(len(rows))); axes[0].set_yticklabels([p for _,p in rows], fontsize=7)
    _tr = mtransforms.blended_transform_factory(axes[0].transAxes, axes[0].transData)
    for fam, e, s in zip(HMMS_KER, fam_edges, fam_sizes):
        axes[0].text(-0.35, e - s/2 - 0.5, fam, fontsize=14, rotation=90, va='center', ha='center', transform=_tr)
    fig.suptitle(f'{MODEL_LABELS[mk]}', fontsize=20, y=0.95)
    sm = ScalarMappable(cmap=cmap, norm=Normalize(-vmax, vmax)); sm.set_array([])
    cb = fig.colorbar(sm, ax=list(axes), fraction=0.025, pad=0.02)
    cb.set_label('$R^2$ after steering, mean over steering layers', fontsize=13); cb.ax.tick_params(labelsize=12)
    tag = 'dtarget' if ref == 'new' else 'dorig'
    fig.savefig(f'{S6_PLOT_DIR}/delta_vs_random_summary_{tag}_{mk}{TAG}.pdf', bbox_inches='tight')
    plt.show(); plt.close()

In [ ]:
# ── Regenerate all Section-6 figures under the current SPLIT ──
ALL_FAMS = ['Mess3','Arch','Wing','Strata']
for mk in MODEL_KEYS:
    print(f'\n######## {MODEL_LABELS[mk]}  [{SPLIT}] ########')
    df = destroy_dfs['belief'].get(mk)
    ker = pd.read_csv(story_path('ntp_matched', mk))    if os.path.exists(story_path('ntp_matched', mk))    else None
    rnd = pd.read_csv(story_path('random_matched', mk)) if os.path.exists(story_path('random_matched', mk)) else None
    # 1. log-NTP: before strips, frozen after-maps (centerpiece), all-HMM summary
    if df is not None:
        logntp_before_strip(df, 'new', mk); logntp_before_strip(df, 'orig', mk)
        logntp_after_map(df, 'new', mk);    logntp_after_map(df, 'orig', mk)
        raw_profile_fig(df, mk, 'log_ntp', ALL_FAMS, "beliefs' log-NTP", 'log-NTP probing layer', 'xmap_summary_rawR2')
    # 2. δ: before strips, after-maps, all-HMM summary
    delta_before_strip(mk, 'new'); delta_before_strip(mk, 'orig')
    delta_after_map(mk, 'new');    delta_after_map(mk, 'orig')
    if ker is not None:
        raw_profile_fig(ker, mk, 'hidden', HMMS_KER, 'δ-belief coordinate', 'δ Belief probing layer', 'ker_summary_rawR2')
    # 3. random: after-maps, all-HMM summary (no before: the projection has nothing to decode pre-steer)
    random_after_map(mk, 'new'); random_after_map(mk, 'orig')
    if rnd is not None:
        raw_profile_fig(rnd, mk, 'rand_proj', HMMS_KER, 'random component', RAND_XLABEL, 'randdir_summary_rawR2', probe='fixed_dir')
    # 3b. side by side: δ (δ steer) vs random component (random steer), all ker-family HMMs
    if ker is not None and rnd is not None:
        delta_vs_random_summary(ker, rnd, mk, ref='new')
        delta_vs_random_summary(ker, rnd, mk, ref='orig')
    # 4. auxiliary log-NTP maps and the all-HMM summary
    if df is not None:
        logntp_after_map(df, 'new', mk, probe='retrain'); logntp_after_map(df, 'orig', mk, probe='retrain')
        raw_profile_fig(df, mk, 'log_ntp', ALL_FAMS, "beliefs' log-NTP", 'log-NTP probing layer', 'xmap_summary_rawR2_retrain',
                        probe='retrain', title_note=' (retrain)')

## Section 7: patching and steering the prediction (Figure 7, Figure 16, Figures 71 to 76)

In [ ]:
# ── Load the Section 7 intervention results (scripts/run_interventions.sh) ──
# Columns: context ('full'), tail_check (the per-sequence relative discrepancy between the cached-prefix pass and the
# full pass, at most about 0.01 for Qwen and Llama and about 0.006 for Gemma), kl_to_*_hmm (the KL restricted to the
# HMM tokens), off_mass, and tgt_kl_to_factual. kl_to_target, kl_to_factual, kl_to_pi_target, and baseline_kl are the
# full-vocabulary KL of Section 4.
FULLCTX_DIR = RESULTS_DIR

def load_fullctx(model_key):
    path = os.path.join(FULLCTX_DIR, f'prediction_interventions_{model_key}.csv')
    if not os.path.exists(path):
        print(f'  MISSING: {path}'); return None
    df = pd.read_csv(path)
    assert (df['context'] == 'full').all(), 'not a full-context file'
    assert set(df['intervention']) == {'none', 'patch', 'steer'}, set(df['intervention'])
    return df

intervene_dfs = {}
for mk in MODEL_KEYS:
    df = load_fullctx(mk)
    if df is not None:
        intervene_dfs[mk] = df
        u = df[df.intervention == 'none']
        print(f"{MODEL_LABELS[mk]}: {len(df):,} rows, {u.groupby('hmm').param.nunique().to_dict()} params/family, "
              f"{u.seed.nunique()} seeds, consistency check max {u.tail_check.max():.3f}")
print(f'\n{len(intervene_dfs)} full-context intervention files loaded')

In [ ]:
# Figures 7 and 16: interventions at full context with the full-vocabulary KL of Section 4 and the steering vector of Section 6.1.
# Solid = KL to the injected target (kl_to_target / kl_to_pi_target for random).
# Dotted = KL to the true/factual belief (kl_to_factual) -- the control showing
# the injection redirects prediction away from the real history.
# Bands = +/-1 SEM across the 10 seeds (on the solid target curves only).
import matplotlib.ticker as mticker

SHOW_FACTUAL = True   # set False to drop the dotted control lines
MAIN_KS = [10, 5, 1]  # rows
SAVE_SUFFIX = ''      # appended to the figure file names intervene_main_{patch,steer}{SAVE_SUFFIX}.pdf

_pc  = '#d94801'   # past-consistent   : dark orange
_pi  = '#fd8d3c'   # past-inconsistent : orange
_rnd = '#8fb3c9'   # random control    : light blue

_CONDS = [
    ('past-consistent',   'past_consistent',   'kl_to_target',    'kl_to_factual', _pc),
    ('past-inconsistent', 'past_inconsistent', 'kl_to_target',    'kl_to_factual', _pi),
    ('random control',    'random',            'kl_to_pi_target', 'kl_to_factual', _rnd),
]

def _istat(d, cond, col, k, interv, layers):
    s = d[(d.condition == cond) & (d.intervention == interv) & (d.k == k)]
    g = s.groupby(['layer', 'seed'])[col].mean().reset_index()
    return g.groupby('layer')[col].agg(['mean', 'sem']).reindex(layers)

def intervention_transpose_figure(df, model_name, interv, targets=TARGETS, ks=MAIN_KS):
    layers = sorted(df[df.layer >= 0].layer.unique())
    fig, axes = plt.subplots(len(ks), len(targets),
                             figsize=(3.25 * len(targets), 2.3 * len(ks)),
                             sharex=True, sharey='col', squeeze=False)
    for ci, (fam, param) in enumerate(targets):
        d = df[(df.hmm == fam) & (df.param == param)]
        assert len(d) > 0, f'no rows for {fam} {param}'
        for ri, k in enumerate(ks):
            ax = axes[ri, ci]
            for zi, (lab, cond, tcol, fcol, color) in enumerate(_CONDS):
                z_solid = 10 - zi
                z_dot   = 7 - zi
                st = _istat(d, cond, tcol, k, interv, layers)
                band = st['sem']
                _slw = 3.0 if cond == 'random' else 3.5
                ax.plot(layers, st['mean'].values, color=color, lw=_slw, zorder=z_solid)
                ax.fill_between(layers, st['mean'] - band, st['mean'] + band,
                                color=color, alpha=0.13, linewidth=0, zorder=2)
                if SHOW_FACTUAL:
                    sf = _istat(d, cond, fcol, k, interv, layers)
                    fband = sf['sem']
                    ax.plot(layers, sf['mean'].values, color=color, lw=2, ls=':', zorder=z_dot)
                    ax.fill_between(layers, sf['mean'] - fband, sf['mean'] + fband,
                                    color=color, alpha=0.10, linewidth=0, zorder=2)
            base = d[d.condition == 'unmodified'].groupby('seed').baseline_kl.mean().mean()
            ax.axhline(base, color='0.5', lw=2.5, ls='-', zorder=1)
            ax.set_yscale('log')
            if fam == 'Arch':
                ax.set_yticks([1, 0.1, 0.01])
            else:
                ax.yaxis.set_major_locator(mticker.LogLocator(base=10.0, numticks=15))
            ax.yaxis.set_minor_locator(mticker.LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1, numticks=100))
            ax.yaxis.set_minor_formatter(mticker.NullFormatter())
            ax.set_xticks([0, 10, 20, 30])
            ax.grid(True, which='major', alpha=0.18, linewidth=0.5)
            ax.grid(False, which='minor')
            if ri == 0:            ax.set_title(fam, fontsize=19, pad=8)
            if ri == len(ks) - 1:  ax.set_xlabel('Layer', fontsize=17)
            if ci == 0:
                ax.set_ylabel('Final Token KL', fontsize=13)
                ax.annotate(f'k = {k}', xy=(0, 0.5), xycoords='axes fraction',
                            xytext=(-78, 0), textcoords='offset points',
                            rotation=90, va='center', ha='center', fontsize=20)
            ax.tick_params(axis='both', labelsize=15, length=3)
    fig.suptitle('Patching' if interv == 'patch' else 'Steering', fontsize=19, y=0.94, x=0.575)
    cond_h = [
        Line2D([], [], color='0.5', lw=4, ls='-', label='Unmodified'),
        Line2D([], [], color=_pc,  lw=4, label='Past-consistent'),
        Line2D([], [], color=_pi,  lw=4, label='Past-inconsistent'),
        Line2D([], [], color=_rnd, lw=4, label='Random control'),
    ]
    leg1 = fig.legend(handles=cond_h, loc='lower center', ncol=4, fontsize=16,
                      frameon=False, bbox_to_anchor=(0.57, -0.01))
    fig.add_artist(leg1)
    if SHOW_FACTUAL:
        style_h = [
            Line2D([], [], linestyle='none', label='Solid Lines: KL of NTP from target sequence'),
            Line2D([], [], linestyle='none', label='Dotted Lines: KL of NTP from original sequence'),
        ]
        fig.legend(handles=style_h, loc='lower center', ncol=2, fontsize=16,
                   frameon=False, bbox_to_anchor=(0.57, -0.055),
                   handlelength=0, handletextpad=0, columnspacing=2.5)
    fig.tight_layout(rect=[0.07, 0.06 if SHOW_FACTUAL else 0.03, 1, 0.97])
    return fig


# %% Render main-text figures for Qwen 3.5 9B (patching + steering) -----
MAIN_MODEL = 'qwen35_9b'
for _interv in ['patch', 'steer']:
    _fig = intervention_transpose_figure(intervene_dfs[MAIN_MODEL], MODEL_LABELS[MAIN_MODEL], _interv)
    _fig.savefig(f'{PLOT_DIR}/intervene_main_{_interv}{SAVE_SUFFIX}.pdf', bbox_inches='tight')
    plt.show(); plt.close(_fig)

In [ ]:
# Appendix P (Figures 71 to 76): the intervention effect for every parametrization, per model, patching and steering,
# faceted by k. The value of a parametrization is the mean over the layers at or beyond 30 percent of the model depth
# of the seed-averaged KL to the target (kl_to_pi_target for the random control), per k. The gray dashed line is the
# model's unintervened KL (baseline_kl); lower means the injection was more successful. The past-consistent and
# past-inconsistent points coincide at k >= 5 for Qwen and Llama, and the k = 1 past-consistent point sits at the
# baseline because its target usually equals the factual belief.
from matplotlib.lines import Line2D

LAYER_FRAC = 0.3   # cutoff = layers >= 30% of model depth (= L>=10 on 32-layer models)
K_LIST = [1, 5, 10]
SAVE_SUFFIX = ''   # appended to the figure file names intervene_all_<model>{SAVE_SUFFIX}.pdf
_SUMM_CONDS = [
    ('Past-cons.',   'past_consistent',   'kl_to_target',    '#d94801'),
    ('Past-incons.', 'past_inconsistent', 'kl_to_target',    '#fd8d3c'),
    ('Random',       'random',            'kl_to_pi_target', '#8fb3c9'),
]
_COND_COLS = [c for *_, c in _SUMM_CONDS]
_INTERVS = [('patch', 'Patching'), ('steer', 'Steering')]

def _param_kl(d, cond, tcol, interv, k, layer_min):
    """Seed-averaged KL-to-injected-target, averaged over layers >= layer_min, given k."""
    s = d[(d.condition == cond) & (d.intervention == interv) & (d.k == k)]
    if len(s) == 0:
        return np.nan
    tgt = s.groupby('layer')[tcol].mean()
    L = [l for l in tgt.index if l >= layer_min]
    return float(tgt.loc[L].mean()) if L else np.nan

for model_key in MODEL_KEYS:
    model_name = MODEL_LABELS.get(model_key, model_key)
    df = intervene_dfs.get(model_key) if 'intervene_dfs' in globals() else load_fullctx(model_key)
    if df is None:
        continue
    df = df[df['hmm'] != 'Spiral']
    layer_min = int(LAYER_FRAC * (df.layer.max() + 1))
    hmms = [h for h in HMM_ORDER if h in df['hmm'].values]
    facets = [(iv, ivn, k) for iv, ivn in _INTERVS for k in K_LIST]

    fig, axes = plt.subplots(len(facets), len(hmms),
                             figsize=(3.3 * len(hmms), 2.9 * len(facets)),
                             squeeze=False)
    x = np.arange(len(_SUMM_CONDS))

    for ri, (interv, interv_name, k) in enumerate(facets):
        for ci, hmm in enumerate(hmms):
            ax = axes[ri, ci]
            params = sorted(df[df['hmm'] == hmm]['param'].unique())
            pcols = plt.get_cmap('Oranges')(np.linspace(0.35, 0.95, len(params)))

            col_vals = {xi: [] for xi in range(len(_SUMM_CONDS))}
            base_vals = []
            for pi, pp in enumerate(params):
                d = df[(df['hmm'] == hmm) & (df['param'] == pp)]
                y = np.array([_param_kl(d, cond, tcol, interv, k, layer_min)
                              for _, cond, tcol, _c in _SUMM_CONDS])
                ax.plot(x, y, '-', color=pcols[pi], lw=1.3, alpha=0.7,
                        marker='o', ms=3, zorder=2)
                for xi, v in enumerate(y):
                    if np.isfinite(v):
                        col_vals[xi].append(v)
                b = d[d.intervention == 'none']['baseline_kl'].mean()
                if np.isfinite(b):
                    base_vals.append(b)

            ymean = [np.nanmean(col_vals[xi]) if col_vals[xi] else np.nan
                     for xi in range(len(_SUMM_CONDS))]
            ax.plot(x, ymean, '-', color='black', lw=3, zorder=3)
            ax.scatter(x, ymean, c=_COND_COLS, s=70, zorder=4,
                       edgecolor='black', linewidth=3)
            if base_vals:
                ax.axhline(np.mean(base_vals), color='0.45', lw=2, ls='--', zorder=1)

            ax.set_ylim(bottom=0)
            ax.set_xticks(x)
            if ri == len(facets) - 1:
                ax.set_xticklabels([lab for lab, *_ in _SUMM_CONDS],
                                   rotation=30, ha='right', fontsize=11)
            else:
                ax.set_xticklabels([])
            ax.set_xlim(-0.3, len(_SUMM_CONDS) - 0.7)
            ax.tick_params(axis='y', labelsize=11, length=3)
            ax.grid(True, axis='y', alpha=0.2, linewidth=0.5); ax.set_axisbelow(True)
            if ri == 0:
                ax.set_title(hmm, fontsize=15)
            if ci == 0:
                ax.set_ylabel(f'{interv_name}, $k$={k}\nKL (interv.$\\to$ target)',
                              fontsize=11)

            if ri == len(facets) - 1:
                handles = [Line2D([], [], color=pcols[pi], lw=2, marker='o', ms=3, label=str(pp))
                           for pi, pp in enumerate(params)]
                handles.append(Line2D([], [], color='black', lw=3, marker='o', ms=5, label='mean'))
                handles.append(Line2D([], [], color='0.45', lw=2, ls='--', label='unintervened KL'))
                ncol = 2 if len(handles) > 6 else 1
                ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.55),
                          ncol=ncol, fontsize=7, frameon=False, handlelength=1.3,
                          columnspacing=1.0, labelspacing=0.3)

    fig.suptitle(f'{model_name} '
                 f'(lower KL: prediction closer to injected target)', fontsize=14, y=1.0)
    plt.tight_layout()
    plt.savefig(f'{PLOT_DIR}/intervene_all_{model_key}{SAVE_SUFFIX}.pdf', bbox_inches='tight')
    plt.show(); plt.close()

## Figure 1 (b), right panel

In [ ]:
# ══ Figure 1 panel: Wing patching at k=10 (standalone, legend below) ══
from matplotlib.lines import Line2D
import matplotlib.ticker as mticker

HMM_F1 = 'Wing'
PARAM_F1 = [p for h, p in TARGETS if h == HMM_F1][0]
K_F1 = 10
INTERV_F1 = 'patch'
PANEL_LW = 5.0

_pc  = '#d94801'   # past-consistent
_pi  = '#fd8d3c'   # past-inconsistent
_rnd = '#8fb3c9'   # random control
_CONDS = [
    ('past-consistent',   'past_consistent',   'kl_to_target',    _pc),
    ('past-inconsistent', 'past_inconsistent', 'kl_to_target',    _pi),
    ('random control',    'random',            'kl_to_pi_target', _rnd),
]

def _istat(d, cond, col, k, interv, layers):
    s = d[(d.condition == cond) & (d.intervention == interv) & (d.k == k)]
    g = s.groupby(['layer', 'seed'])[col].mean().reset_index()
    return g.groupby('layer')[col].agg(['mean', 'sem']).reindex(layers)

df = intervene_dfs[MAIN_MODEL]
d = df[(df.hmm == HMM_F1) & (df.param == PARAM_F1)]
layers = sorted(df[df.layer >= 0].layer.unique())

fig, ax = plt.subplots(figsize=(8.5, 3.7))
fig.patch.set_facecolor('white')

for zi, (lab, cond, tcol, color) in enumerate(_CONDS):
    z_solid = 10 - zi
    st = _istat(d, cond, tcol, K_F1, INTERV_F1, layers)
    _slw = 5.5 if cond == 'random' else 6.0
    ax.plot(layers, st['mean'].values, color=color, lw=_slw, zorder=z_solid)
    ax.fill_between(layers, st['mean'] - st['sem'], st['mean'] + st['sem'],
                    color=color, alpha=0.13, linewidth=0, zorder=2)

base = d[d.condition == 'unmodified'].groupby('seed').baseline_kl.mean().mean()
ax.axhline(base, color='0.5', lw=4.5, ls='-', zorder=1)

ax.set_yscale('log')
ax.yaxis.set_major_locator(mticker.LogLocator(base=10.0, numticks=15))
ax.yaxis.set_minor_locator(mticker.LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1, numticks=100))
ax.yaxis.set_minor_formatter(mticker.NullFormatter())
ax.set_xticks([0, 10, 20, 30])
ax.grid(True, which='both', alpha=0.18, linewidth=0.5)
ax.tick_params(axis='both', labelsize=30, length=7, width=2.5)
ax.tick_params(axis='y', which='minor', length=4, width=2.0)
ax.set_xlabel('Layer', fontsize=34)
ax.set_ylabel('Final Token KL', fontsize=31)
for spine in ax.spines.values():
    spine.set_linewidth(PANEL_LW)

legend_handles = [
    Line2D([], [], color='0.5', lw=8, ls='-', label='Unmodified'),
    Line2D([], [], color=_pc,  lw=8, label='Past-consistent'),
    Line2D([], [], color=_pi,  lw=8, label='Past-inconsistent'),
    Line2D([], [], color=_rnd, lw=8, label='Random control'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=2,
           fontsize=30, frameon=False, bbox_to_anchor=(0.5, -0.60),
           handletextpad=0.5, handlelength=1.4, labelspacing=0.35)

plt.savefig(f'{PLOT_DIR}/fig1_intervene_wing.svg', bbox_inches='tight', dpi=300)
plt.show(); plt.close()